In [ ]:
!apt-get install -y fonts-nanum > /dev/null 2>&1
!pip install lightgbm shap joblib -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import joblib

from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, roc_auc_score,
                             precision_recall_curve, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)
import lightgbm as lgb

fm._load_fontmanager(try_read_cache=False)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
df = pd.read_excel('batter_stats_all.xlsx')

print(f"전체 shape: {df.shape}")
print(f"\n[지명여부 분포]")
print(df['지명여부'].value_counts())

print(f"\n[연도별 지명 현황]")
print(df.groupby('year').apply(lambda x: pd.Series({
    '전체': len(x),
    '지명': (x['지명여부']=='O').sum()
})))

In [ ]:
# 1) 비율 지표 결측 → 평균 대체
ratio_cols = ['AVG', 'OBP', 'SLG', 'OPS',
              'IsoP', 'IsoD', 'K%', 'BB%', 'BB/K', 'BABIP',
              'GB%', 'LD%', 'FB%', 'ifFB%', 'ofFB%', 'GB/FB', 'HR/FB%',
              'DIR_L%', 'DIR_LC%', 'DIR_C%', 'DIR_RC%', 'DIR_R%',
              'Bunt%', 'if1B%',
              'SP_AVG', 'SP_OBP', 'SP_SLG', 'SP_OPS',
              'wOBA', 'wRC', 'RC27', 'PF_weighted', 'wRAA', 'wRC+']
for col in ratio_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].mean())

# 2) player_id 결측 → 임의 ID
n_missing = df['player_id'].isnull().sum()
df.loc[df['player_id'].isnull(), 'player_id'] = range(
    900_000_000_000,
    900_000_000_000 + n_missing
)

# 결측치 재확인
remaining = df.drop(columns=['라운드','순위','포지션','지명구단']).isnull().sum()
print("처리 후 결측:")
print(remaining[remaining > 0] if remaining.sum() > 0 else "✅ 결측 없음")

In [ ]:
df = df.sort_values(['player_id', 'year']).reset_index(drop=True)

# 통산 누적
df['total_PA']  = df.groupby('player_id')['PA'].transform('sum')
df['total_H']   = df.groupby('player_id')['H'].transform('sum')
df['total_HR']  = df.groupby('player_id')['HR'].transform('sum')
df['total_RBI'] = df.groupby('player_id')['RBI'].transform('sum')
df['total_R']   = df.groupby('player_id')['R'].transform('sum')
df['total_G']   = df.groupby('player_id')['G'].transform('sum')

# 시즌 평균
df['avg_AVG'] = df.groupby('player_id')['AVG'].transform('mean')
df['avg_OPS'] = df.groupby('player_id')['OPS'].transform('mean')
df['avg_OBP'] = df.groupby('player_id')['OBP'].transform('mean')

# 최고 성적 (피크 능력)
df['best_OPS']  = df.groupby('player_id')['OPS'].transform('max')
df['best_AVG']  = df.groupby('player_id')['AVG'].transform('max')
df['best_wRC+'] = df.groupby('player_id')['wRC+'].transform('max')

# 활동 시즌 수
df['seasons_played'] = df.groupby('player_id')['year'].transform('count')

print("✅ 누적 피처 13개 추가")
print(df[['total_PA','avg_OPS','best_OPS','seasons_played']].head())

In [ ]:
features = [
    # 기본 카운트 (8)
    'G', 'PA', 'AB', 'H', '2B', '3B', 'HR', 'BB',
    # 출루/삼진 (4)
    'IBB', 'HBP', 'SO', 'SF',
    # 득점/주루 (4)
    'RBI', 'R', 'SB', 'CS',
    # 비율 스탯 (4)
    'AVG', 'OBP', 'SLG', 'OPS',
    # 세이버매트릭스 (7)
    'IsoP', 'IsoD', 'K%', 'BB%', 'BABIP',
    'wOBA', 'wRC+',
    # 종합 지표 (3)
    'RE24', 'RC27', 'wRAA',
    # 타구 분포 (4)
    'GB%', 'LD%', 'FB%', 'HR/FB%',
    # 클러치 (4)
    'SP_PA', 'SP_H', 'SP_AVG', 'SP_OPS',
    # 다년도 누적 (13)
    'total_PA', 'total_H', 'total_HR', 'total_RBI', 'total_R', 'total_G',
    'avg_AVG', 'avg_OPS', 'avg_OBP',
    'best_OPS', 'best_AVG', 'best_wRC+',
    'seasons_played',
]

df['지명_binary'] = (df['지명여부'] == 'O').astype(int)

X = df[features]
y = df['지명_binary']
groups = df['player_id']

print(f"피처 수: {len(features)}개")
print(f"X shape: {X.shape}")
print(f"지명자: {y.sum()}명 / 전체 {len(y)}명 ({y.mean()*100:.2f}%)")

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('model', lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=6,
        num_leaves=31,
        min_child_samples=5,
        reg_alpha=0.1,
        reg_lambda=0.1,
        class_weight='balanced',
        random_state=42,
        verbose=-1
    ))
])

oof_proba = cross_val_predict(
    pipeline, X, y,
    cv=sgkf.split(X, y, groups=groups),
    method='predict_proba',
    n_jobs=-1
)[:, 1]

print(f"AUROC (CV): {roc_auc_score(y, oof_proba):.4f}")

In [ ]:
precision_arr, recall_arr, thresholds = precision_recall_curve(y, oof_proba)

f1_arr = np.where(
    (precision_arr[:-1] + recall_arr[:-1]) == 0, 0,
    2 * precision_arr[:-1] * recall_arr[:-1]
      / (precision_arr[:-1] + recall_arr[:-1])
)

best_idx = f1_arr.argmax()
best_threshold = thresholds[best_idx]
best_f1 = f1_arr[best_idx]

print(f"최적 Threshold: {best_threshold:.3f}")
print(f"최적 F1-score : {best_f1:.4f}")

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(recall_arr, precision_arr, color='#378ADD', lw=2)
axes[0].scatter(recall_arr[best_idx], precision_arr[best_idx],
                color='#D85A30', s=120, zorder=5,
                label=f'최적 threshold = {best_threshold:.2f}')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(thresholds, f1_arr, color='#1D9E75', lw=2)
axes[1].axvline(best_threshold, color='#D85A30', ls='--',
                label=f'threshold = {best_threshold:.2f}')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('F1-score')
axes[1].set_title('Threshold별 F1-score', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
y_pred = (oof_proba >= best_threshold).astype(int)

print("=" * 50)
print("[ 타자 드래프트 예측 모델 — 최종 성능 ]")
print("=" * 50)

auroc  = roc_auc_score(y, oof_proba)
f1_mac = f1_score(y, y_pred, average='macro')
f1_pos = f1_score(y, y_pred, pos_label=1)

print(f"AUROC          : {auroc:.4f}")
print(f"F1 (macro)     : {f1_mac:.4f}")
print(f"F1 (지명 클래스): {f1_pos:.4f}")
print(f"Threshold      : {best_threshold:.3f}")

# Precision@K
def precision_at_k(y_true, y_proba, k):
    top_k = np.argsort(y_proba)[::-1][:k]
    return np.array(y_true)[top_k].sum()

print(f"\nPrecision@10 : {precision_at_k(y, oof_proba, 10)}명 / 10명")
print(f"Precision@20 : {precision_at_k(y, oof_proba, 20)}명 / 20명")
print(f"Precision@50 : {precision_at_k(y, oof_proba, 50)}명 / 50명")

print("\n" + "=" * 50)
print(classification_report(y, y_pred, target_names=['미지명', '지명']))

# 혼동 행렬
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['미지명', '지명'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion Matrix (threshold={best_threshold:.2f})', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
pipeline.fit(X, y)
model = pipeline.named_steps['model']

label_kor = {
    # 기본 카운트
    'G':'경기수','PA':'타석수','AB':'타수','H':'안타',
    '2B':'2루타','3B':'3루타','HR':'홈런','BB':'볼넷',
    # 출루/삼진
    'IBB':'고의사구','HBP':'사구','SO':'삼진','SF':'희생플라이',
    # 득점/주루
    'RBI':'타점','R':'득점','SB':'도루','CS':'도루실패',
    # 비율 스탯
    'AVG':'타율','OBP':'출루율','SLG':'장타율','OPS':'OPS',
    # 세이버매트릭스
    'IsoP':'ISO(순수장타)','IsoD':'IsoD(순수출루)',
    'K%':'K%','BB%':'BB%','BABIP':'BABIP',
    'wOBA':'wOBA','wRC+':'wRC+',
    # 종합 지표
    'RE24':'RE24','RC27':'RC27','wRAA':'wRAA',
    # 타구 분포
    'GB%':'땅볼%','LD%':'라인드라이브%','FB%':'뜬공%','HR/FB%':'HR/FB%',
    # 클러치
    'SP_PA':'클러치 타석','SP_H':'클러치 안타',
    'SP_AVG':'클러치 타율','SP_OPS':'클러치 OPS',
    # 다년도 누적
    'total_PA':'통산 타석','total_H':'통산 안타','total_HR':'통산 홈런',
    'total_RBI':'통산 타점','total_R':'통산 득점','total_G':'통산 경기수',
    'avg_AVG':'평균 타율','avg_OPS':'평균 OPS','avg_OBP':'평균 출루율',
    'best_OPS':'최고 OPS','best_AVG':'최고 타율','best_wRC+':'최고 wRC+',
    'seasons_played':'활동 시즌 수',
}

imp_df = (pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
})
.sort_values('importance', ascending=True)
.tail(15))

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh([label_kor[f] for f in imp_df['feature']],
        imp_df['importance'], color='#1D9E75', edgecolor='none', height=0.65)
ax.set_title('타자 모델 — LightGBM 피처 중요도 Top 15', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance')
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
joblib.dump(pipeline,       'batter_model.pkl')
joblib.dump(best_threshold, 'batter_threshold.pkl')
joblib.dump(features,       'batter_features.pkl')

print("✅ 타자 모델 저장 완료")
print("  - batter_model.pkl")
print("  - batter_threshold.pkl")
print("  - batter_features.pkl")